# Introduction to Large Language Models
### [Yoav Ram](http://yoavram.com)

A graduate workshop on how large language models are built, from a permutation-invariant set model to a GPT you pretrain, align and then hand a shell. Everything is written in **pure-functional JAX** — no Flax, no Equinox — so every model is a `params` dict and a `forward` function, and nothing important happens inside a framework you cannot read.

It picks up where [DataSciPy](https://github.com/yoavram/DataSciPy) leaves off.

## What you will be able to do

- **Match an architecture to the structure of the data** — recognise permutation invariance, and say what a flattened network has to learn that Deep Sets and a Set Transformer get for free.
- **Implement, in JAX, from scratch:** an RNN, a GRU, a character-level transformer, a BPE tokenizer, and a GPT with RMSNorm, RoPE, QK-norm and causal attention.
- **Explain what attention buys** over a recurrent state, and support the explanation with a controlled experiment you ran rather than a table you were shown.
- **Train a tokenizer and reason about it quantitatively** — compression ratio, bits per character, and what happens to out-of-distribution text.
- **Pretrain a 26 M-parameter language model** and read its loss curve, its evaluation, and its hyperparameter sensitivity.
- **Align a pretrained model**: supervised fine-tuning with a response-masked loss, then GRPO against a verifiable reward — group-relative advantages, a clipped surrogate objective, a KL penalty.
- **Detect reward hacking.** Compute what the task permits *before* training, put the ceiling next to the result, and read the model's output — the three things that caught it here when every statistic looked immaculate.
- **Generate text deliberately**: temperature, top-*k*, top-*p*, and a fixed-shape jitted decode loop that compiles once instead of once per token.
- **Build a tool-using agent** on a local model, and argue about its safety concretely: least privilege, a gate that fails closed, and a prompt injection that actually lands.

## How to read a notebook here

Each one is a lesson with a single argument, and the code is part of the argument rather than an appendix to it. Exercises are at the end; the ★ ones need a design decision of your own. **Results live in the notebooks** — this page is navigation only, and deliberately quotes no numbers.

## The arc of the workshop

Each notebook adds one thing the previous one could not do. Read them in this order.

**Sets → what an architecture assumes.** A poker hand is a *set*: five cards in any order are the same hand. Feeding it to a flat network means the network has to learn that, from data, badly. Deep Sets and the Set Transformer build the assumption into the architecture instead. This is the idea the rest of the workshop runs on — an architecture is a claim about the structure of the data.

**Recurrence → memory.** Text is not a set; order is the whole point. An RNN carries a state vector forward one character at a time, so in principle it remembers everything and in practice it remembers very little. The GRU adds gates that let the state be carried unchanged, which is what makes long-range memory trainable.

**Attention → memory without a bottleneck.** A recurrent state is a fixed-size summary of the past, written before you know which question will be asked of it. Attention drops the summary and looks directly at every earlier position. The three character-level notebooks train on the same corpus, budget and seeds, so this comparison is one you produce rather than one you are told.

**Tokenisation → a vocabulary worth modelling.** Characters are a cheap vocabulary and an expensive sequence: everything is long, and the model spends its capacity on spelling. BPE learns a vocabulary from the corpus and roughly halves the sequence length — which matters because attention costs *O*(*T*²) — and it introduces the fact that a model's "units" are a design decision with measurable costs on both sides.

**Pretraining → a real language model.** A 26 M-parameter GPT in pure JAX — RMSNorm, RoPE, QK-norm, causal attention — trained on TinyStories. This notebook carries a second story alongside the first: a GPT has many hyperparameters and an unobvious sensitivity to them, so how do you decide? The QK-norm measurements are there to open that discussion, not to close it.

**Alignment → doing what was asked.** A pretrained model continues text; it does not follow instructions. SFT teaches the instruction format with a response-masked loss. GRPO then optimises a *reward* rather than a likelihood — and, in this notebook, promptly games it: the policy beats the ground-truth ceiling by stuffing the required words in where they make no sense. That failure is the lesson, and the fix — an LLM judge the hack cannot satisfy by construction — is the second one.

**Inference → using the thing.** Sampling knobs (temperature, top-*k*, top-*p*), why a naive generation loop is slow and what a fixed-shape jitted one costs, and the same prompt through every checkpoint so the alignment story can be read instead of scored.

**Agents → a model with hands.** A ReAct loop over a local LLM, a bash tool, and the safety argument that follows from letting a language model run commands: least privilege, an approval gate that fails closed, and a working prompt injection against the pipeline you built two cells earlier.

## The notebooks

Times are wall-clock for a full top-to-bottom run, measured on one **RTX A4000** from the committed outputs. They are what it costs to *reproduce* a notebook; in a session you normally load the checkpoint instead and the notebook takes minutes.

| # | Notebook | Hardware | Full run | Needs first |
|---|---|---|---|---|
| 1 | [sets.ipynb](sets.ipynb) | GPU | ~47 min | — |
| 2 | [RNN.ipynb](RNN.ipynb) | GPU | ~57 min | — |
| 3 | [GRU.ipynb](GRU.ipynb) | GPU | ~78 min | 2 (shared config) |
| 4 | [text-transformer.ipynb](text-transformer.ipynb) | GPU | ~22 min | 2 (shared config) |
| 5 | [bpe-tokenizer.ipynb](bpe-tokenizer.ipynb) | CPU | ~1 min | — |
| 6 | [nanochat.ipynb](nanochat.ipynb) | GPU | ~2 h 10 min | 5, then `build_train_artifacts.py` |
| — | [nanochat-exercises.ipynb](nanochat-exercises.ipynb) | GPU | — | 6 |
| 7 | [nanochat-sft.ipynb](nanochat-sft.ipynb) | GPU | ~13 min | 6 |
| 8 | [nanochat-grpo.ipynb](nanochat-grpo.ipynb) | GPU + Ollama | ~80 min | 7 |
| 9 | [nanochat-chat.ipynb](nanochat-chat.ipynb) | GPU | ~5 min | 6, 7, 8 |
| 10 | [minisweagent.ipynb](minisweagent.ipynb) | CPU + Ollama | interactive | — |

### Set models

- **[sets.ipynb](sets.ipynb)** — Flattened FFN, Deep Sets and a Set Transformer on poker-hand classification. Permutation invariance as an architectural assumption; a majority-class baseline and a task ceiling, because on this data accuracy alone will tell you a model is excellent when it has never once predicted a rare hand.

### Character-level language models

These three share one controlled experiment — same corpus (`data/shakespear3.txt`), split, budget, learning-rate schedule, state policy and seeds — defined by `RNN.ipynb` and written to `checkpoints/charlm_config.json`. Each notebook ends with the cumulative results table, so the comparison is something you produce rather than something asserted here. Run `RNN.ipynb` first; the other two read its config.

- **[RNN.ipynb](RNN.ipynb)** — a vanilla recurrent network in JAX, character by character: state, `jax.lax.scan`, and why a fixed-size carry is both the mechanism and the limit. Defines the shared experiment and the unigram/bigram baselines.
- **[GRU.ipynb](GRU.ipynb)** — gates on top of that state. What an update gate at `z → 0` does to the gradient path, and what the gates cost in parameters.
- **[text-transformer.ipynb](text-transformer.ipynb)** — the same task with attention instead of recurrence, plus depth and context-length sweeps.

### BPE tokenizer

- **[bpe-tokenizer.ipynb](bpe-tokenizer.ipynb)** — Byte Pair Encoding written from scratch: the merge loop, corpus cleaning, vocabulary analysis, compression ratio and bits per character, and what the encoder falls back to on text it has never seen. Emits the shared `bpe.py` and saves `checkpoints/bpe_tokenizer.pkl`. Runs on the 22.5 MB validation split by default; `CORPUS = 'train'` switches it to the 2.23 GB split.

### Pretraining

- **[nanochat.ipynb](nanochat.ipynb)** — a 26 M-parameter GPT in pure JAX: RMSNorm, RoPE, QK-norm, causal attention, trained on TinyStories. Builds the model cell by cell and emits those cells as `nanochat_model.py`, which every later notebook imports. Also measures three QK-norm variants and hands the hyperparameter question to the room. Saves `checkpoints/nanochat_checkpoint.pkl` and `nanochat_best.pkl`.
- **[nanochat-exercises.ipynb](nanochat-exercises.ipynb)** — exercises against the pretrained checkpoint: a scaling sweep, LayerNorm against RMSNorm, per-sample gradients with `vmap`, and a few larger projects.

### Supervised fine-tuning

- **[nanochat-sft.ipynb](nanochat-sft.ipynb)** — instruction tuning on TinyStories-Instruct with a response-masked loss: constrained story generation from `Words:`/`Features:` prompts, `<|endoftext|>` as the turn boundary, and a constraint-satisfaction metric that checks the output rather than its likelihood. Saves `checkpoints/nanochat_sft_best.pkl` and `sft_examples.npz`.

### Reinforcement learning from verifiable rewards

- **[nanochat-grpo.ipynb](nanochat-grpo.ipynb)** — Group Relative Policy Optimization: group sampling, group-relative advantages, a clipped surrogate loss, a KL penalty. Trains twice — once on a word-matching reward, which the policy games by beating the ground-truth ceiling, and once on that reward times an LLM judge, which mostly fixes it. Needs Ollama for the judge. Saves `checkpoints/nanochat_grpo_checkpoint.pkl` (words only) and `nanochat_grpo_judge_checkpoint.pkl` (words × judge).

### Inference

- **[nanochat-chat.ipynb](nanochat-chat.ipynb)** — run the trained models: sampling (temperature, top-*k*, top-*p*), a fixed-shape jitted generation loop measured against the naive one, and the same prompt through all four checkpoints so the reward hack and its fix can be read rather than scored. Trains nothing.

### Agents

- **[minisweagent.ipynb](minisweagent.ipynb)** — a minimal LLM agent on Ollama: the ReAct loop, a bash tool, system-prompt specialisation, a human approval gate that denies when it cannot ask, and a live prompt-injection demonstration. **Run this one interactively** — it executes shell commands you approve one at a time, which is why it ships with its outputs cleared.

## Prerequisites

- **Python**, NumPy and Matplotlib at working speed, and enough deep learning to know what a loss, a gradient step and a training loop are. [DataSciPy](https://github.com/yoavram/DataSciPy) is the assumed background.
- **JAX is taught as we go** — `jit`, `grad`, `vmap`, `scan`, PRNG keys and pytrees each appear where they are first needed. No prior JAX required.
- **An NVIDIA GPU** for notebooks 1–4 and 6–9. The times above are for a 16 GB RTX A4000; anything comparable will do. On CPU or Apple silicon the tokenizer and agent notebooks are fine and the training ones are not.
- **Ollama** with a local model, for the GRPO judge and the agent notebook.

## What depends on what

`sets.ipynb` stands alone. The three character notebooks share one experiment: run `RNN.ipynb` first — it writes `checkpoints/charlm_config.json` and the split that the other two read.

The nanochat chain passes artefacts forward, so each notebook needs the previous one to have been run at least once (or its checkpoint to be sitting in `checkpoints/` already):

```
bpe-tokenizer.ipynb  →  bpe.py  +  checkpoints/bpe_tokenizer.pkl        (valid split, 22.5 MB)
        ↓
build_train_artifacts.py  →  bpe_tokenizer_train.pkl + train_tokens.npy (train split, 2.23 GB)
        ↓
nanochat.ipynb       →  nanochat_model.py, nanochat_best.pkl
        ↓
nanochat-sft.ipynb   →  nanochat_sft_best.pkl, sft_examples.npz
        ↓
nanochat-grpo.ipynb  →  nanochat_grpo_checkpoint.pkl, nanochat_grpo_judge_checkpoint.pkl
        ↓
nanochat-chat.ipynb  →  (nothing — it reads all of the above and compares them)
```

Two modules, **`bpe.py` and `nanochat_model.py`, are generated by the notebook that teaches them** and imported by everything downstream, so there is exactly one copy of the model code and you watched it being written. Both are tracked in git; re-running a notebook rewrites them, and a dirty diff there is expected rather than a mistake.

The one step that is a script rather than a notebook is `build_train_artifacts.py`: pretraining uses the 2.23 GB train split, and putting 1.06 billion tokens through a pure-Python BPE takes ~16 minutes — too slow to sit in the middle of a lesson, and it is memory management rather than teaching. It imports its tokenizer from `bpe.py`, so it is the notebook's own code, run once offline. A tokenizer and the corpus it was fitted to are one artefact, which is why pretraining does not reuse the validation-split tokenizer.

Datasets download themselves on first run — TinyStories (22.5 MB validation), TinyStories-Instruct (26.9 MB) and the UCI poker hands (24 MB), or all of them at once with `python download_data.py`. Shakespeare is already in `data/`. See [`README.md`](README.md) for the full list.

## Setup

The full setup procedure lives in [`README.md`](README.md) — follow it there. In short:

```bash
pixi install          # creates the environment (JAX, NumPy, Matplotlib, Pandas, optax, JupyterLab)
pixi run jupyter lab  # launches JupyterLab inside it
```

On Linux `pixi install` pulls `jax[cuda12]`, which needs an NVIDIA driver ≥ 525 (check with `nvidia-smi`). On macOS it installs CPU-only JAX.

Two notebooks — `nanochat-grpo.ipynb` (the judge) and `minisweagent.ipynb` (the agent) — call a local model over [Ollama](https://ollama.com):

```bash
ollama pull qwen3.5:9b    # 6.6 GB; qwen3.5:4b (3.4 GB) is the low-VRAM fallback
```

## Jupyter tips

- **Autocompletion**: press `Tab` to complete variable names or, after a dot, to browse methods and attributes.
- **Documentation**: press `Shift+Tab` inside a function call to show its arguments and docstring.
- **Terminal**: `File → New → Terminal` opens a shell inside JupyterLab.
- **Kernel**: if two notebooks compete for the GPU, shut down unused kernels from the sidebar (square-in-circle icon). JAX takes the whole device, so two training notebooks at once will fail on memory rather than share it.

### Monitoring CPU

Open a terminal and run `htop`.

### Monitoring GPU

```bash
nvidia-smi          # one-off
watch -n 1 nvidia-smi
```